# 🔱 Shiv AI Voice Cloning
**Owner: Shri Ram Nag | PAISAWALA Channel**

Model: `Shriramnag/Shiv-AI-Voice-Cloning` from HuggingFace

> ⚡ GPU runtime use karein: Runtime → Change runtime type → T4 GPU

In [ ]:
# ✅ STEP 1: GPU Check
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️ GPU nahi mila! Runtime → Change runtime type → T4 GPU select karein')

In [ ]:
# ✅ STEP 2: Dependencies Install
!pip install -q gradio huggingface_hub transformers accelerate
!pip install -q scipy numpy
print('✅ Dependencies installed!')

In [ ]:
# ✅ STEP 3: HuggingFace Se Model Download
import os
from huggingface_hub import snapshot_download

REPO_ID = 'Shriramnag/Shiv-AI-Voice-Cloning'
LOCAL_DIR = './Shiv-AI-Voice-Cloning'

if not os.path.exists(LOCAL_DIR):
    print(f'📥 Downloading model from HuggingFace: {REPO_ID}')
    print('⏳ Please wait... (2.45 GB model.safetensors + other files)')
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=LOCAL_DIR,
        local_dir_use_symlinks=False
    )
    print('✅ Model downloaded successfully!')
else:
    print('✅ Model already exists locally, skipping download.')

# Downloaded files check
print('\n📂 Downloaded files:')
for f in os.listdir(LOCAL_DIR):
    size = os.path.getsize(os.path.join(LOCAL_DIR, f)) / 1e6
    print(f'  {f} — {size:.2f} MB')

In [ ]:
# ✅ STEP 4: Path Setup & Imports
import sys
import logging
import re
import uuid
import numpy as np
import torch
import gradio as gr

# Model path add to system
MODEL_LOCAL_PATH = './Shiv-AI-Voice-Cloning'
sys.path.insert(0, MODEL_LOCAL_PATH)

# Import model classes
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

try:
    from subtitle import subtitle_maker
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except ImportError:
    WHISPER_LANGUAGE_CODE = None
    print('Note: subtitle module partial import')

print('✅ Imports successful!')

In [ ]:
# ✅ STEP 5: Model Load
print('🔱 Shiv AI Voice Cloning Model load ho raha hai...')

model = OmniVoice.from_pretrained(
    MODEL_LOCAL_PATH,
    device_map='cuda',
    dtype=torch.float16,
    load_asr=False,
)

sampling_rate = model.sampling_rate
print(f'✅ Model loaded! Sampling rate: {sampling_rate} Hz')

In [ ]:
# ✅ STEP 6: Core Functions
temp_audio_dir = './Shiv_Audio'
os.makedirs(temp_audio_dir, exist_ok=True)

EVENT_TAGS = [
    '[laughter]', '[sigh]', '[confirmation-en]', '[question-en]',
    '[surprise-wa]', '[dissatisfaction-hnn]'
]

INSERT_TAG_JS = """
(tag_val, current_text) => {
    const textarea = document.querySelector('.shiv-textbox textarea');
    if (!textarea) return current_text + ' ' + tag_val;
    const start = textarea.selectionStart;
    const end = textarea.selectionEnd;
    return current_text.slice(0, start) + ' ' + tag_val + ' ' + current_text.slice(end);
}
"""

def _gen_core(text, language, ref_audio, mode, ref_text=None):
    if not text or not text.strip():
        return None, '⚠️ कृपया टेक्स्ट लिखें।'

    gen_config = OmniVoiceGenerationConfig(
        num_step=32,
        guidance_scale=2.0,
        denoise=True,
        preprocess_prompt=True,
        postprocess_output=True,
    )

    kw = dict(
        text=text.strip(),
        language=language if language != 'Auto' else None,
        generation_config=gen_config
    )

    if mode == 'clone' and ref_audio:
        kw['voice_clone_prompt'] = model.create_voice_clone_prompt(
            ref_audio=ref_audio, ref_text=ref_text
        )

    audio = model.generate(**kw)
    waveform = (audio[0] * 32767).astype(np.int16)
    return (sampling_rate, waveform), '✅ सफलतापूर्वक जनरेट हुआ!'

print('✅ Functions ready!')

In [ ]:
# ✅ STEP 7: Gradio UI Launch
theme = gr.themes.Soft(primary_hue='orange', font=['Inter', 'Arial', 'sans-serif'])

css = """
.gradio-container {max-width: 100% !important;}
footer {visibility: hidden !important;}
.shiv-header {text-align: center; margin: 20px auto; padding: 10px; border-bottom: 2px solid #ff6600;}
.tag-btn {background: #fff3e0 !important; border: 1px solid #ffcc80 !important; color: #e65100 !important;}
"""

with gr.Blocks(theme=theme, css=css, title='Shiv AI Voice Cloning') as demo:
    gr.HTML("""
        <div class='shiv-header'>
            <h1 style='font-size: 2.5em; color: #ff6600; margin-bottom: 0;'>🔱 Shiv AI Voice Cloning</h1>
            <p style='font-size: 1.1em; color: #555;'>Advanced Multilingual Speech Engine | <b>Owner: Shri Ram Nag</b></p>
            <p style='font-size: 0.85em; color: #888;'>Model: Shriramnag/Shiv-AI-Voice-Cloning | 646 Languages | PAISAWALA</p>
        </div>
    """)

    with gr.Tabs():
        with gr.TabItem('🎙️ Voice Clone (आवाज़ क्लोनिंग)'):
            with gr.Row():
                with gr.Column():
                    vc_text = gr.Textbox(
                        label='टेक्स्ट लिखें (Text to Synthesize)',
                        lines=5,
                        elem_classes='shiv-textbox',
                        placeholder='यहाँ अपना हिंदी/English टेक्स्ट लिखें...'
                    )
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            btn = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            btn.click(fn=None, inputs=[btn, vc_text], outputs=vc_text, js=INSERT_TAG_JS)

                    vc_lang = gr.Dropdown(
                        label='भाषा (Language)',
                        choices=['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES),
                        value='Auto'
                    )
                    vc_ref_audio = gr.Audio(
                        label='🎤 Reference Audio (अपनी आवाज़ अपलोड करें)',
                        type='filepath'
                    )
                    vc_status = gr.Textbox(label='Status', interactive=False)
                    vc_btn = gr.Button('🔱 Shiv AI से आवाज़ बनाएं', variant='primary', size='lg')

                with gr.Column():
                    vc_audio = gr.Audio(label='🔊 Shiv AI Output', type='numpy')
                    gr.Markdown('### **प्रोजेक्ट डेवलपर: श्री राम नाग**')
                    gr.Markdown('यह सिस्टम आपकी आवाज़ को **646+ भाषाओं** में क्लोन कर सकता है।')
                    gr.Markdown('📦 Model: `Shriramnag/Shiv-AI-Voice-Cloning`')

    gr.HTML("<div style='text-align:center;padding:20px;color:#888;'>© 2026 Shiv AI Voice Cloning | Shri Ram Nag | PAISAWALA</div>")

    def start_clone(text, lang, ref_aud):
        res, status = _gen_core(text, lang, ref_aud, mode='clone')
        return res, status

    vc_btn.click(fn=start_clone, inputs=[vc_text, vc_lang, vc_ref_audio], outputs=[vc_audio, vc_status])

demo.launch(share=True, debug=False)
print('🔱 Shiv AI launched! Upar wala public URL copy karein.')